In [3]:
"""
Evaluation script.
Loads a trained model and runs episodes to measure performance.
Optionally records video.
"""

import numpy as np
import torch
from config import Config
from env import make_env
from agent import DoubleDQNAgent

In [4]:
def evaluate(model_path, n_episodes=10, render=False, record=False):
    """
    Load a trained agent and run evaluation episodes.

    Args:
        model_path: str, path to saved model weights
        n_episodes: int, number of evaluation episodes
        render: bool, whether to render the environment
        record: bool, whether to record video

    Returns:
        list of episode rewards
    """

    config = Config()

    env, preprocessor, stacker = make_env(
        config.ENV_NAME,
        config.FRAME_SIZE,
        config.FRAME_STACK
    )

    n_actions = env.action_space.n

    device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    agent = DoubleDQNAgent(
        n_frames=config.FRAME_STACK,
        n_actions=n_actions,
        config=config,
        device=device
    )

    # Load trained weights
    agent.load(model_path)

    # Set network to evaluation mode
    agent.online_net.eval()

    episode_rewards = []

    for episode in range(n_episodes):

        obs, info = env.reset()

        state = stacker.reset(
            preprocessor.process(obs)
        )

        done = False
        episode_reward = 0.0

        while not done:

            if render:
                env.render()

            # Greedy action selection
            state_tensor = (
                torch.FloatTensor(state)
                .unsqueeze(0)
                .to(device)
            )

            with torch.no_grad():
                q_values = agent.online_net(state_tensor)
                action = q_values.argmax(dim=1).item()

            next_obs, reward, terminated, truncated, info = env.step(action)

            done = terminated or truncated

            next_state = stacker.append(
                preprocessor.process(next_obs)
            )

            episode_reward += reward

            state = next_state

        episode_rewards.append(episode_reward)

        print(
            f"Episode {episode + 1}/{n_episodes} "
            f"Reward: {episode_reward:.2f}"
        )

    env.close()

    return episode_rewards

In [7]:
rewards = evaluate(
    model_path="checkpoint_500000.pth",
    n_episodes=10,
    render=False
)

print("Mean reward:", np.mean(rewards))

Episode 1/10 Reward: -33.00
Episode 2/10 Reward: -33.00
Episode 3/10 Reward: -33.00
Episode 4/10 Reward: -33.00
Episode 5/10 Reward: -33.00
Episode 6/10 Reward: -33.00
Episode 7/10 Reward: -33.00
Episode 8/10 Reward: -33.00
Episode 9/10 Reward: -33.00
Episode 10/10 Reward: -33.00
Mean reward: -33.0


In [5]:
if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("--model", type=str, required=True)
    parser.add_argument("--episodes", type=int, default=10)
    parser.add_argument("--render", action="store_true")
    parser.add_argument("--record", action="store_true")
    args = parser.parse_args()

    rewards = evaluate(args.model, args.episodes, args.render, args.record)
    print(f"Mean reward over {len(rewards)} episodes: {np.mean(rewards):.2f}")
    print(f"Std: {np.std(rewards):.2f}")

usage: ipykernel_launcher.py [-h] --model MODEL [--episodes EPISODES] [--render] [--record]
ipykernel_launcher.py: error: the following arguments are required: --model


SystemExit: 2

C:\Users\divya\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
